# model-archival — Google Drive Archiver

Downloads open-source LLM/LRM weights from HuggingFace directly into Google Drive.

---

### Fire-and-forget workflow

1. **First time:** run `setup.ipynb` once to copy library files to Drive
2. **Each session:** edit Cell 1 if needed → **Runtime → Run all** → close browser
3. **Next day:** open notebook → **Runtime → Run all** — it resumes automatically

Progress is saved after **every file** — a crash or timeout wastes at most one file.

---

| Cell | Purpose | Re-run when |
|------|---------|-------------|
| 1 | Configuration | Changing tiers/models |
| 2 | Install deps + load token | New session |
| 3 | Mount Drive | New session |
| 4 | Load library + state | New session |
| 5 | Build download queue | New session |
| **6** | **▶ Run downloads** | **Every session** |
| 7 | Quick resume (skip 2–5) | Resuming after interrupt |
| 8 | Verify archive integrity | After downloads |
| 9 | Status summary | Any time |

---
## Cell 1 — Configuration
Edit once. Values persist for the session.

In [ ]:
# ── Storage ───────────────────────────────────────────────────────────────────
# Root folder on your Google Drive (created automatically if missing)
DRIVE_ROOT = "/content/drive/MyDrive/model-archive"

# ── What to download ─────────────────────────────────────────────────────────
# Tiers: 'A' (raw BF16 general), 'B' (raw BF16 code), 'C' (GGUF quantized), 'D' (uncensored)
# Use 'ALL' for everything, or a list: ['C', 'D']
TIERS = ['C', 'D']   # Start small — these complete in 1–2 sessions

# Priority: 1 = token-free only  |  2 = include gated models (needs HF_TOKEN)
MAX_PRIORITY = 2

# Override to download specific models only (list of HF repo IDs)
# e.g. ['deepseek-ai/DeepSeek-R1-Distill-Qwen-32B']
SPECIFIC_MODELS = []

# ── Session settings ─────────────────────────────────────────────────────────
# Stop queuing new models this many hours before Colab's 24h limit
# (already-started models will finish; protects against mid-model cutoff)
SESSION_WARN_HOURS = 1.5

# Set False to actually download; True to preview the queue only
DRY_RUN = False

# ── Advanced ─────────────────────────────────────────────────────────────────
# Colab session limit in hours (free=12, Pro/Pro+=24)
SESSION_LIMIT_HOURS = 24

---
## Cell 2 — Install deps & load HF token

**Add your token as a Colab Secret before running:**
🔑 key icon → **Add new secret** → Name: `HF_TOKEN` → toggle **Notebook access**

In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '--upgrade', 'huggingface_hub', 'pyyaml'])

# Load HF token — Colab Secrets first, then environment variable
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    print(f'✓ HF_TOKEN loaded ({len(HF_TOKEN)} chars)')
else:
    print('⚠  HF_TOKEN not set — gated models (Llama, Gemma, Mistral Large) will fail')
    print('   Token-free models (DeepSeek, Qwen, Phi-4, all Tier C/D) will still work')

print('✓ Dependencies ready')

---
## Cell 3 — Mount Google Drive
A browser popup will ask you to sign in.

In [ ]:
import shutil, time
from pathlib import Path
from google.colab import drive

def mount_drive(force_remount: bool = False) -> bool:
    """Mount Drive, retrying once if the first attempt fails."""
    for attempt in range(2):
        try:
            drive.mount('/content/drive', force_remount=(attempt > 0 or force_remount))
            # Verify mount is actually usable
            test = Path('/content/drive/MyDrive')
            test.mkdir(parents=True, exist_ok=True)
            _ = list(test.iterdir())  # will throw if not mounted
            return True
        except Exception as e:
            if attempt == 0:
                print(f'Drive mount attempt 1 failed ({e}), retrying...')
                time.sleep(5)
            else:
                print(f'Drive mount failed: {e}')
                return False

DRIVE_MOUNTED = mount_drive()

if DRIVE_MOUNTED:
    DRIVE_PATH = Path(DRIVE_ROOT)
    DRIVE_PATH.mkdir(parents=True, exist_ok=True)
    (DRIVE_PATH / 'logs').mkdir(exist_ok=True)

    STATE_FILE = DRIVE_PATH / 'run_state.json'
    INDEX_FILE = DRIVE_PATH / 'global_index.jsonl'
    LOG_DIR    = DRIVE_PATH / 'logs'

    total, used, free = shutil.disk_usage('/content/drive/MyDrive')
    print(f'✓ Drive mounted')
    print(f'  Total : {total/1024**4:.2f} TB')
    print(f'  Used  : {used/1024**4:.2f} TB  ({used/total*100:.1f}%)')
    print(f'  Free  : {free/1024**4:.2f} TB')
    print(f'  Root  : {DRIVE_ROOT}')

    if free < 100 * 1024**3:
        print('⚠  Less than 100 GB free!')
else:
    print('✗ Drive not mounted — cannot continue')

---
## Cell 4 — Load library, state & session guard

In [ ]:
import json, sys, importlib, time
from pathlib import Path
from datetime import datetime, timezone

# Locate colab/lib — tries repo clone then Drive copy
LIB_CANDIDATES = [
    Path('/content/model-archival/colab/lib'),
    Path('/content/drive/MyDrive/model-archival/colab/lib'),
    Path('/content/lib'),
]
for candidate in LIB_CANDIDATES:
    if candidate.exists() and (candidate / 'downloader.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        print(f'✓ Library: {candidate}')
        break
else:
    raise FileNotFoundError(
        'Cannot find colab/lib/downloader.py.\n'
        'Run setup.ipynb first to copy files to Drive.'
    )

import downloader as dl
importlib.reload(dl)

# Load or initialise model-level state
if STATE_FILE.exists():
    state = json.loads(STATE_FILE.read_text())
    complete = sum(1 for v in state.values() if v == 'complete')
    print(f'✓ State: {len(state)} models tracked, {complete} complete')
else:
    state = {}
    print('✓ State: fresh (no previous run)')

def save_state():
    """Atomically persist state to Drive."""
    tmp = STATE_FILE.with_suffix('.tmp')
    tmp.write_text(json.dumps(state, indent=2) + '\n')
    tmp.replace(STATE_FILE)

# Session time guard — warns before Colab cuts the session
SESSION_START = time.monotonic()
guard = dl.SessionGuard(limit_hours=SESSION_LIMIT_HOURS)
print(f'✓ Session guard: limit={SESSION_LIMIT_HOURS}h  warn={SESSION_WARN_HOURS}h before limit')
print('✓ Library and state ready')

---
## Cell 5 — Load registry & build download queue

In [ ]:
import yaml

REGISTRY_CANDIDATES = [
    Path('/content/model-archival/local/config/registry.yaml'),
    Path('/content/drive/MyDrive/model-archival/local/config/registry.yaml'),
    Path('/content/registry.yaml'),
]
for c in REGISTRY_CANDIDATES:
    if c.exists():
        registry_path = c
        break
else:
    raise FileNotFoundError('Cannot find registry.yaml — run setup.ipynb first')

with open(registry_path) as f:
    registry = yaml.safe_load(f)

all_models = registry.get('models', [])

if SPECIFIC_MODELS:
    queue = [m for m in all_models if m['id'] in SPECIFIC_MODELS]
elif TIERS == 'ALL':
    queue = [m for m in all_models if m['priority'] <= MAX_PRIORITY]
else:
    tiers = [TIERS] if isinstance(TIERS, str) else TIERS
    queue = [m for m in all_models
             if m['tier'] in tiers and m['priority'] <= MAX_PRIORITY]

queue.sort(key=lambda m: (m['priority'], m['tier'], m['id']))

n_done    = sum(1 for m in queue if state.get(m['id']) == 'complete')
n_partial = sum(1 for m in queue if state.get(m['id']) == 'partial')
n_todo    = len(queue) - n_done

print(f'Registry : {len(all_models)} models total')
print(f'Queue    : {len(queue)} matching  |  {n_done} complete  {n_partial} partial  {n_todo} remaining')
print()

COL_ICONS = {'complete': '✓', 'partial': '~', 'failed': '✗',
             'in_progress': '⟳', 'pending': '○', None: '○'}

for m in queue:
    s    = state.get(m['id'])
    icon = COL_ICONS.get(s, '?')
    auth = '  [TOKEN REQUIRED]' if m.get('requires_auth') and not HF_TOKEN else ''
    print(f'  {icon} [{m["tier"]}] P{m["priority"]}  {m["id"]}{auth}')

---
## Cell 6 — ▶ Run downloads

**This is the main cell.** Re-run this cell (or the whole notebook) each session — it always resumes.

- Progress saved after **every file** — crash/timeout safe
- Stops queuing new models when < `SESSION_WARN_HOURS` remain
- Enable **Runtime → Background execution** (Pro+) then close the browser

In [ ]:
import shutil, time
from datetime import datetime, timezone
from IPython.display import clear_output, display
import ipywidgets as widgets

# ── Progress display ──────────────────────────────────────────────────────────
def make_progress_bar(label: str, total: int):
    """Returns (bar_widget, update_fn). Works in Colab."""
    bar  = widgets.IntProgress(value=0, min=0, max=max(total, 1),
                               layout=widgets.Layout(width='80%'))
    lbl  = widgets.Label(value=label)
    box  = widgets.HBox([lbl, bar])
    display(box)

    def update(n: int, status: str = ''):
        bar.value = n
        lbl.value = f'{label}  {n}/{total}  {status}'
        if n >= total:
            bar.bar_style = 'success'

    return update

# ── Drive health check ────────────────────────────────────────────────────────
def check_drive() -> bool:
    try:
        _ = list(Path('/content/drive/MyDrive').iterdir())
        return True
    except Exception:
        return False

def ensure_drive():
    """Re-mount Drive if it has become unavailable."""
    if not check_drive():
        print('⚠  Drive became unavailable — remounting...')
        from google.colab import drive as _d
        _d.mount('/content/drive', force_remount=True)
        time.sleep(5)
        if check_drive():
            print('✓ Drive remounted successfully')
        else:
            raise RuntimeError('Drive unavailable after remount — cannot continue')

# ─────────────────────────────────────────────────────────────────────────────

if DRY_RUN:
    print('DRY RUN — no files will be downloaded\n')
    for m in queue:
        s = state.get(m['id'], 'pending')
        print(f'  [{m["tier"]}] P{m["priority"]}  {m["id"]}  → {s}')

else:
    session_start = datetime.now(timezone.utc)
    print(f'Session started : {session_start.strftime("%Y-%m-%d %H:%M UTC")}')
    print(f'Archive root    : {DRIVE_ROOT}')
    print(f'Models in queue : {len(queue)}')
    print(f'Session limit   : {SESSION_LIMIT_HOURS}h  (stops new models with {SESSION_WARN_HOURS}h left)')
    print()

    session_results = []
    session_bytes   = 0

    for idx, model_cfg in enumerate(queue, 1):
        mid    = model_cfg['id']
        tier   = model_cfg.get('tier', 'A')
        quants = model_cfg.get('quant_levels')
        tok    = HF_TOKEN if model_cfg.get('requires_auth') else None

        print(f'── [{idx}/{len(queue)}] {mid} ─────────────────────────────────')
        print(f'   {guard.status_line()}')

        # Check Drive is still usable before each model
        ensure_drive()

        # Build per-model progress callback
        _prog_update = [None]  # list so closure can assign

        def on_file_progress(filename, status, files_done, files_total,
                             _pu=_prog_update):
            if _pu[0]:
                short = filename.split('/')[-1]
                _pu[0](files_done, f'{status}  {short}')

        # Only create progress bar once we know file count
        # (downloader logs file count in BEGIN line — this is fine)

        result = dl.download_model(
            model_id=mid,
            dest_root=DRIVE_PATH,
            hf_token=tok,
            tier=tier,
            quant_levels=quants,
            global_index_path=INDEX_FILE,
            state=state,
            save_state_fn=save_state,
            session_guard=guard,
            on_file_progress=on_file_progress,
        )
        session_results.append(result)
        session_bytes += result.total_bytes

        status_icon = {'complete': '✓', 'skipped': '✓', 'partial': '~',
                       'failed': '✗'}.get(result.status, '?')
        print(f'   {status_icon} {result.status.upper()}  {result.total_bytes/1024**3:.2f} GB'
              + (f'  {result.elapsed_seconds/60:.1f} min' if result.elapsed_seconds else ''))
        print()

        # Stop if session limit approaching
        if guard.is_near_limit(SESSION_WARN_HOURS):
            print(f'⚠  {guard.status_line()}')
            print('⚠  Stopping queue. Start a new session to continue.')
            break

    # ── Session summary ───────────────────────────────────────────────────────
    n_ok      = sum(1 for r in session_results if r.status in ('complete', 'skipped'))
    n_partial = sum(1 for r in session_results if r.status == 'partial')
    n_fail    = sum(1 for r in session_results if r.status == 'failed')

    print('═' * 60)
    print('SESSION SUMMARY')
    print(f'  {n_ok} complete/skipped  {n_partial} partial  {n_fail} failed')
    print(f'  {session_bytes/1024**3:.2f} GB transferred this session')
    print(f'  {guard.status_line()}')
    print(f'  State file: {STATE_FILE}')
    print('═' * 60)

    if n_fail:
        print()
        print('Failed models (will retry next session):')
        for r in session_results:
            if r.status == 'failed':
                print(f'  ✗ {r.model_id}: {r.error}')

    # Write a brief session log to Drive
    ts = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    log_path = LOG_DIR / f'session-{ts}.md'
    log_path.write_text(
        f'# Session {ts}\n\n'
        f'- Started: {session_start.isoformat()}\n'
        f'- Models: {n_ok} OK  {n_partial} partial  {n_fail} failed\n'
        f'- Transferred: {session_bytes/1024**3:.2f} GB\n'
        f'- {guard.status_line()}\n'
    )
    print(f'\nSession log: {log_path}')

---
## Cell 7 — Quick resume (skip Cells 2–5)

**Use this when resuming mid-session** after a kernel restart or interrupt.
Assumes Drive is already mounted and lib is already in `sys.path`.
Otherwise run the full notebook from Cell 2.

In [ ]:
# ── Quick-resume cell ─────────────────────────────────────────────────────────
# Run this cell alone to restart downloads after a kernel restart,
# without re-running setup, mount, or registry cells.

import json, sys, importlib, time, os
from pathlib import Path

_needs_full_setup = False

# Check Drive is accessible
try:
    _ = list(Path('/content/drive/MyDrive').iterdir())
except Exception:
    print('Drive not mounted — running full mount...')
    from google.colab import drive
    drive.mount('/content/drive')

# Check all required names exist in this kernel
_required = ['DRIVE_PATH', 'STATE_FILE', 'INDEX_FILE', 'LOG_DIR',
             'queue', 'state', 'save_state', 'guard', 'HF_TOKEN',
             'DRY_RUN', 'SESSION_LIMIT_HOURS', 'SESSION_WARN_HOURS', 'dl']
_missing = [n for n in _required if n not in dir()]
if _missing:
    print(f'Missing from kernel: {_missing}')
    print('Run the full notebook (Runtime → Run all) instead of this cell.')
else:
    importlib.reload(dl)
    # Refresh state from Drive
    if STATE_FILE.exists():
        state.update(json.loads(STATE_FILE.read_text()))
    complete = sum(1 for v in state.values() if v == 'complete')
    remaining = sum(1 for m in queue if state.get(m['id']) != 'complete')
    print(f'✓ State refreshed: {complete} complete, {remaining} remaining in queue')
    print(f'✓ {guard.status_line()}')
    print()
    print('Re-run Cell 6 to continue downloads.')

---
## Cell 8 — Verify archive integrity

Cross-checks every `.sha256` sidecar against `manifest.json`.
Run after downloads, or any time to audit the archive.
Set `REHASH = True` for a full re-hash from disk (very slow — only needed after hardware events).

In [ ]:
import hashlib, json
from pathlib import Path
from datetime import datetime, timezone

REHASH         = False   # True = re-hash every file from disk
FAILURES_ONLY  = False   # True = only print failures

def verify_archive(root: Path, rehash: bool = False, failures_only: bool = False):
    manifests = sorted(root.rglob('manifest.json'))
    if not manifests:
        print('No manifests found — nothing to verify.')
        return

    total_f = ok_f = missing_f = mismatch_f = 0
    report  = [
        '# Archive Verification Report\n',
        f'Generated : {datetime.now(timezone.utc).isoformat()}',
        f'Root      : {root}',
        f'Mode      : {"full re-hash" if rehash else "sidecar cross-check"}',
        ''
    ]

    for mf in manifests:
        model_dir = mf.parent
        try:
            manifest = json.loads(mf.read_text())
        except Exception as e:
            print(f'  ✗ CORRUPT MANIFEST  {mf}  ({e})')
            continue

        mid   = manifest.get('model_id', str(model_dir))
        files = manifest.get('files', {})
        m_ok = m_missing = m_mm = 0

        for fn, fdata in files.items():
            fpath    = model_dir / fn
            expected = fdata.get('sha256', '')
            total_f += 1

            if not fpath.exists():
                print(f'  ✗ MISSING    {mid}/{fn}')
                missing_f += 1; m_missing += 1
                continue

            if rehash:
                h = hashlib.sha256()
                with open(fpath, 'rb') as f:
                    while chunk := f.read(8 * 1024 * 1024):
                        h.update(chunk)
                actual = h.hexdigest()
            else:
                sc     = Path(str(fpath) + '.sha256')
                actual = sc.read_text().strip() if sc.exists() else None

            if actual is None:
                print(f'  ? NO_SIDECAR {mid}/{fn}')
                m_mm += 1; mismatch_f += 1
            elif actual != expected:
                print(f'  ✗ MISMATCH   {mid}/{fn}')
                m_mm += 1; mismatch_f += 1
            else:
                m_ok += 1; ok_f += 1

        icon = '✓' if (m_missing + m_mm) == 0 else '✗'
        line = f'{icon} {mid}  {m_ok} OK  {m_missing} missing  {m_mm} mismatch'
        if not failures_only or (m_missing + m_mm) > 0:
            print(line)
        report.append(f'## {mid}\n\n{m_ok} OK | {m_missing} missing | {m_mm} mismatch\n')

    print()
    print(f'Total: {total_f} files  {ok_f} OK  {missing_f} missing  {mismatch_f} mismatch')

    ts          = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    report_path = LOG_DIR / f'verify-report-{ts}.md'
    report_path.write_text('\n'.join(report))
    print(f'Report saved: {report_path}')


verify_archive(DRIVE_PATH, rehash=REHASH, failures_only=FAILURES_ONLY)

---
## Cell 9 — Status summary

In [ ]:
import json, shutil
from pathlib import Path

# Refresh state from Drive in case another session updated it
if STATE_FILE.exists():
    state = json.loads(STATE_FILE.read_text())

by_status: dict[str, list[str]] = {}
for mid, s in state.items():
    by_status.setdefault(s, []).append(mid)

ICONS = {'complete': '✓', 'partial': '~', 'failed': '✗',
         'in_progress': '⟳', 'pending': '○'}

print(f'Archive root : {DRIVE_ROOT}')
print(f'State file   : {STATE_FILE}')
print(f'{guard.status_line()}')
print()

for label in ('complete', 'partial', 'in_progress', 'failed', 'pending'):
    models = by_status.get(label, [])
    if models:
        icon = ICONS.get(label, '?')
        print(f'{icon} {label.upper()} ({len(models)}):')
        for m in sorted(models):
            print(f'    {m}')
        print()

# Models in queue not yet in state
unseen = [m['id'] for m in queue if m['id'] not in state]
if unseen:
    print(f'○ NOT STARTED ({len(unseen)}):')
    for m in unseen:
        print(f'    {m}')
    print()

_, _, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free   : {free/1024**4:.2f} TB')

# Recent session logs
logs = sorted(LOG_DIR.glob('session-*.md'), reverse=True)[:5]
if logs:
    print()
    print('Recent sessions:')
    for log in logs:
        print(f'  {log.name}')